In [1]:
import sys
import os
import json
from pathlib import Path

sys.path.append(os.path.abspath("N:\\CancerEpidem\\BrBreakthrough\\DeliveryProcess\\Schema_and_Derivation_utils\\Histopathology\\scripts"))
from histopath_loader_utils import load_histopath_df
from histopath_building_utils import build_histopath_records
from pseudo_anon_histopath import apply_histopath_privacy_transforms, write_pipeline_pseudoanon_schema

sys.path.append(os.path.abspath("N:\\CancerEpidem\\BrBreakthrough\\DeliveryProcess\\Schema_and_Derivation_utils"))
from utilities import createLogger
from config import live_server, Delivery_log_path

sys.path.append(os.path.abspath("N:\\CancerEpidem\\BrBreakthrough\\DeliveryProcess\\Schema_and_Derivation_utils\\Questionnaire\\R0\\scripts"))
from common_utils import validate_data, save_output


In [2]:
logger = createLogger("Histopathology", Delivery_log_path)


In [3]:
schema_base = r"N:\\CancerEpidem\\BrBreakthrough\\DeliveryProcess\\Schema_and_Derivation_utils\\Histopathology\\schemas"
SCHEMA_PATH = schema_base + "\\" + r"raw\\HistoPath_BrCa.json"

target_base = r"N:\\CancerEpidem\\BrBreakthrough\\DeliveryProcess\\Schema_and_Derivation_utils\\CancerSummary\\json_schemas"
target_path = target_base + "\\" + r"NewCancerSummary.json"


In [4]:
def run_histopath_brca(server: str, pathology_db: str = "UpLoads", table_name: str = "Histopath_BrCa_GS_v1"):
    logger = createLogger("HistoPath_BrCa", r"N:\CancerEpidem\BrBreakthrough\DeliveryProcess\Logs")

    schema, df, missing_schema_fields = load_histopath_df(
        schema_path=SCHEMA_PATH,
        server=server,
        logger=logger,
        pathology_db=pathology_db,
        table_name=table_name,
    )

    records = build_histopath_records(schema, df)

    logger.info(f"Histopath_BrCa_GS_v1 rows: {0 if df is None else len(df):,}")
    logger.info(f"Output records:             {len(records):,}")

    if missing_schema_fields:
        logger.warning(
            "Schema variables missing from SQL table: "
            + ", ".join(missing_schema_fields)
        )
    else:
        logger.info("All schema variables were found in the SQL table.")

    return records, missing_schema_fields


In [5]:
records, missing_schema_fields = run_histopath_brca(server=live_server)


2026-03-27 10:06:10 - INFO: Loaded SQL columns: 87
2026-03-27 10:06:10 - INFO: Loaded SQL columns: 87
2026-03-27 10:06:10 - INFO: Loaded SQL columns: 87
2026-03-27 10:06:10 - INFO: Matched schema columns kept: 65
2026-03-27 10:06:10 - INFO: Matched schema columns kept: 65
2026-03-27 10:06:10 - INFO: Matched schema columns kept: 65
2026-03-27 10:06:10 - INFO: SQL columns not present in schema and therefore dropped after loading: DateCreated, Edat, DiagSource, DCISGrowthPatternOther, TypeOtherMalignantTumour, TypeOtherTumour, TypeComponentOther, SingleNodePositivity, SiteofOtherNodes, TStageMod, NStageMod, MStageMod, RStage, ER_Score, PR_Score, HER2_Type, CK56_Status, CK56_Score, CK56_Type, AlternativeSource, AlternativeSourceDate, GeneralComments
2026-03-27 10:06:10 - INFO: SQL columns not present in schema and therefore dropped after loading: DateCreated, Edat, DiagSource, DCISGrowthPatternOther, TypeOtherMalignantTumour, TypeOtherTumour, TypeComponentOther, SingleNodePositivity, Siteo

In [6]:
records_pa = apply_histopath_privacy_transforms(records, server=live_server, logger=logger)


In [7]:
write_pipeline_pseudoanon_schema(
    in_source_schema_path=SCHEMA_PATH,
    in_target_schema_path = target_path ,
    out_schema_path=schema_base + "\\" + r"pseudo_anon\\HistoPath_BrCa_PseudoAnon.json"
)

'N:\\\\CancerEpidem\\\\BrBreakthrough\\\\DeliveryProcess\\\\Schema_and_Derivation_utils\\\\Histopathology\\\\schemas\\pseudo_anon\\\\HistoPath_BrCa_PseudoAnon.json'

In [8]:
with open(os.path.join(schema_base + "\\" + r"pseudo_anon", "HistoPath_BrCa_PseudoAnon.json"), 'r') as schema:
    schema_new = json.load(schema)

In [9]:
# Validate pseudo-anonymised output against pseudo-anon schema
pseudo_schema_path = schema_base + "\\" + r"pseudo_anon\\HistoPath_BrCa_PseudoAnon.json"
with open(pseudo_schema_path, "r") as f:
    pseudo_schema = json.load(f)
validate_data(records_pa, pseudo_schema, schema_path=pseudo_schema_path)


Validating 3,657 items...
100% - Validation completed in 3.98 seconds
✓ All items are valid


In [10]:
save_output(records_pa, f"HistoPath_BrCa", logger, stage="s5_derived")

2026-03-27 10:06:25 - INFO: Saved output to N:\CancerEpidem\BrBreakthrough\DeliveryProcess\Data_Output_Testing\s5_derived\HistoPath_BrCa.json
2026-03-27 10:06:25 - INFO: Saved output to N:\CancerEpidem\BrBreakthrough\DeliveryProcess\Data_Output_Testing\s5_derived\HistoPath_BrCa.json
2026-03-27 10:06:25 - INFO: Saved output to N:\CancerEpidem\BrBreakthrough\DeliveryProcess\Data_Output_Testing\s5_derived\HistoPath_BrCa.json


Output saved: N:\CancerEpidem\BrBreakthrough\DeliveryProcess\Data_Output_Testing\s5_derived\HistoPath_BrCa.json
